In [8]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

CRS_WGS84 = "EPSG:4326"
CRS_SRC = "EPSG:6668"
CRS_ANALYSIS = "EPSG:6678"

data = Path("data")

# -------------------------
# 1. Administrative boundaries
# -------------------------
adm1 = gpd.read_file(data / "gadm41_JPN_1.shp").to_crs(CRS_ANALYSIS)
adm2 = gpd.read_file(data / "gadm41_JPN_2.shp").to_crs(CRS_ANALYSIS)

tochigi = adm1[adm1["NAME_1"].str.contains("Tochigi|栃木", na=False)]
utsunomiya = adm2[adm2["NAME_2"].str.contains("Utsunomiya|宇都宮", na=False)]

# LRTは芳賀町にも伸びるため、corridor分析では芳賀も含める
corridor_muni = adm2[
    adm2["NAME_2"].str.contains("Utsunomiya|Haga|宇都宮|芳賀", na=False)
]

# -------------------------
# 2. Population mesh
# -------------------------
pop = gpd.read_file(data / "250m_mesh_2024_09.shp", encoding="cp932")
pop = pop.set_crs(CRS_SRC, allow_override=True).to_crs(CRS_ANALYSIS)

pop = gpd.clip(pop, corridor_muni)

keep_pop_cols = ["MESH_ID", "SHICODE", "PT00_2025", "PTC_2025", "RTC_2025", "geometry"]
pop = pop[[c for c in keep_pop_cols if c in pop.columns]]

pop["elderly_share_2025"] = pop["PTC_2025"] / pop["PT00_2025"]
pop["centroid"] = pop.geometry.centroid

# -------------------------
# 3. Bus stops
# -------------------------
bus = gpd.read_file(data / "P11-22_09.shp", encoding="cp932")
bus = bus.set_crs(CRS_SRC, allow_override=True).to_crs(CRS_ANALYSIS)
bus = gpd.clip(bus, corridor_muni)

bus["stop_name_clean"] = (
    bus["P11_001"].astype(str)
    .str.replace(" ", "", regex=False)
    .str.replace("　", "", regex=False)
    .str.replace("停留所", "", regex=False)
)

# 近接・同名の重複チェック用
bus["x_round"] = bus.geometry.x.round(-1)
bus["y_round"] = bus.geometry.y.round(-1)
bus_dupes = bus.duplicated(subset=["stop_name_clean", "x_round", "y_round"], keep=False)

# -------------------------
# 4. Rail lines and stations
# -------------------------
rail_line = gpd.read_file(data / "N05-25_RailroadSection2.shp", encoding="cp932")
rail_sta = gpd.read_file(data / "N05-25_Station2.shp", encoding="cp932")

rail_line = rail_line.set_crs(CRS_SRC, allow_override=True).to_crs(CRS_ANALYSIS)
rail_sta = rail_sta.set_crs(CRS_SRC, allow_override=True).to_crs(CRS_ANALYSIS)

rail_line = gpd.clip(rail_line, corridor_muni)
rail_sta = gpd.clip(rail_sta, corridor_muni)

# 同一駅名・複数路線の dissolve
rail_sta["station_name_clean"] = (
    rail_sta["N05_011"].astype(str)
    .str.replace(" ", "", regex=False)
    .str.replace("　", "", regex=False)
)
rail_sta_dissolved = rail_sta.dissolve(by="station_name_clean", as_index=False)
rail_sta_dissolved["geometry"] = rail_sta_dissolved.geometry.centroid

# -------------------------
# 5. LRT stops
# -------------------------
# 実務上は公式停留場座標または手作成CSVを用意する
lrt_stops = gpd.read_file(data / "lrt_stops.shp").to_crs(CRS_ANALYSIS)

lrt_stops["buf_500m"] = lrt_stops.geometry.buffer(500)
lrt_stops["buf_800m"] = lrt_stops.geometry.buffer(800)

buf500 = gpd.GeoDataFrame(
    lrt_stops[["stop_name"]],
    geometry=lrt_stops["buf_500m"],
    crs=CRS_ANALYSIS
)

buf800 = gpd.GeoDataFrame(
    lrt_stops[["stop_name"]],
    geometry=lrt_stops["buf_800m"],
    crs=CRS_ANALYSIS
)

# -------------------------
# 6. Population in LRT walking catchments
# -------------------------
pop_in_500 = gpd.overlay(pop, buf500, how="intersection")
pop_summary_500 = (
    pop_in_500
    .groupby("stop_name")[["PT00_2025", "PTC_2025"]]
    .sum()
    .reset_index()
)

pop_summary_500["elderly_share_in_500m"] = (
    pop_summary_500["PTC_2025"] / pop_summary_500["PT00_2025"]
)

# -------------------------
# 7. Bus-LRT connectivity
# -------------------------
bus_near_lrt = gpd.sjoin(bus, buf500, how="inner", predicate="within")

bus_connectivity = (
    bus_near_lrt
    .groupby("stop_name")
    .agg(
        bus_stop_count=("P11_001", "count"),
        operator_count=("P11_002", "nunique")
    )
    .reset_index()
)

# -------------------------
# 8. Land price
# -------------------------
land = gpd.read_file(data / "L01-26_09.shp", encoding="cp932")
land = land.set_crs(CRS_SRC, allow_override=True).to_crs(CRS_ANALYSIS)
land = gpd.clip(land, corridor_muni)

land_keep = [
    "L01_001", "L01_002", "L01_007", "L01_008", "L01_009",
    "L01_024", "L01_025", "L01_048", "L01_050", "geometry"
]
land = land[[c for c in land_keep if c in land.columns]]

# LRT最寄距離
lrt_union = lrt_stops.geometry.union_all()
land["dist_to_lrt_m"] = land.geometry.distance(lrt_union)

land["lrt_zone"] = pd.cut(
    land["dist_to_lrt_m"],
    bins=[0, 500, 1000, 2000, np.inf],
    labels=["0-500m", "500-1000m", "1000-2000m", "2000m+"]
)

land_summary = (
    land
    .groupby("lrt_zone", observed=True)
    .agg(
        n=("L01_008", "count"),
        mean_price=("L01_008", "mean"),
        median_price=("L01_008", "median"),
        mean_yoy_change=("L01_009", "mean")
    )
    .reset_index()
)

# -------------------------
# 9. Road data
# -------------------------
road = gpd.read_file(data / "N13-24_5439.shp", encoding="cp932")
road = road.set_crs(CRS_SRC, allow_override=True).to_crs(CRS_ANALYSIS)
road = gpd.clip(road, corridor_muni)

road["length_m"] = road.geometry.length

road_density = road["length_m"].sum() / (corridor_muni.geometry.area.sum() / 1_000_000)

print("Road density km/km2:", road_density / 1000)

DataSourceError: data/lrt_stops.shp: No such file or directory